# 🔤 Tokenizers: How LLMs Read Text

Exploring how large language models convert raw text into numerical token sequences using [Hugging Face Transformers](https://huggingface.co/docs/transformers).

**What this notebook covers:**
- How tokenizers encode and decode text
- Comparing tokenizer vocabularies across different LLMs (Llama 3.1, Phi-4, DeepSeek, QwenCoder)
- How chat templates format multi-turn conversations before they reach a model
- The key insight: LLMs only ever see sequences of integers, never raw text

**Models used:** `meta-llama/Meta-Llama-3.1-8B`, `microsoft/Phi-4-mini-instruct`, `deepseek-ai/DeepSeek-V3.1`, `Qwen/Qwen2.5-Coder-7B-Instruct`

> **Requirements:** Python 3.10+, a [Hugging Face account](https://huggingface.co), and access approved for Llama 3.1 (free — see setup below).


## 1. Setup

### 1.1 Install dependencies

In [ ]:
# Uncomment and run if packages are not yet installed
# !pip install -q datasets==3.6.0 transformers==4.57.6 python-dotenv huggingface_hub


### 1.2 Imports

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import AutoTokenizer


### 1.3 Authenticate with Hugging Face

Store your HF token in a `.env` file at the project root:

```
HUGGING_FACE_TOKEN=hf_your_token_here
```

Get your token at: https://huggingface.co/settings/tokens (requires read permissions).


In [ ]:
load_dotenv(override=True)
hf_token = os.getenv("HUGGING_FACE_TOKEN")

if hf_token and hf_token.startswith("hf_"):
    print("✅ HuggingFace token found")
else:
    raise EnvironmentError("HF token not found. Set HUGGING_FACE_TOKEN in your .env file.")

login(hf_token, add_to_git_credential=True)


### 1.4 Llama 3.1 Access

Meta requires accepting their terms of service before using Llama models.

1. Visit: https://huggingface.co/meta-llama/Meta-Llama-3.1-8B
2. Accept the license at the top of the page (uses the same email as your HF account)
3. Approval is typically granted within a few minutes

Once approved for any Llama 3.1 model, access applies to the full 3.1 family.


---
## 2. Basic Tokenization with Llama 3.1

In [ ]:
# Load the base (non-instruct) Llama 3.1 tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B",
    trust_remote_code=True
)


In [ ]:
# Encode a sample sentence into token IDs
text = "I am excited to show Tokenizers in action to my LLM engineers"
tokens = tokenizer.encode(text)
print("Token IDs:", tokens)


In [ ]:
# Compare character, word, and token counts
character_count = len(text)
word_count = len(text.split())
token_count = len(tokens)

print(f"Characters : {character_count}")
print(f"Words      : {word_count}")
print(f"Tokens     : {token_count}")
print(f"\nRatio: ~{token_count / word_count:.2f} tokens per word")


In [ ]:
# Decode the entire token sequence back to the original string
print("Decoded full string:")
print(tokenizer.decode(tokens))


In [ ]:
# Decode each token individually to see the sub-word fragments
print("Individual token fragments:")
print(tokenizer.batch_decode(tokens))


### 2.1 Vocabulary

In [ ]:
# Inspect tokens that were added on top of the base vocabulary (e.g. special tokens)
added = tokenizer.get_added_vocab()
print(f"Added vocab size: {len(added)}")
print("Sample added tokens:", list(added.items())[:10])


In [ ]:
# Total vocabulary size
print(f"Total vocabulary size: {len(tokenizer.vocab):,} tokens")


---
## 3. Instruct Models and Chat Templates

Many models have an **Instruct** variant fine-tuned for conversational use. These models expect prompts in a specific format with system, user, and assistant roles.

`apply_chat_template()` converts a standard messages list into the correctly formatted string for each model.


In [ ]:
# Load the Instruct variant of Llama 3.1
tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    trust_remote_code=True
)


In [ ]:
# A standard messages list (OpenAI-compatible format)
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
]

# Convert to the model's expected prompt format
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)


### 💡 Key Insight: What LLMs Actually Receive

It's easy to assume LLMs accept Python dictionaries — but they don't. Every LLM is fundamentally a sequence model operating on integers.

The transformation pipeline is:

```
messages (Python dicts)
    → formatted string (via chat template)
    → token fragments (tokenization)
    → token IDs (integer sequence)  ← this is the actual model input
```

> **The input to an LLM is a flat sequence of integer token IDs.**  
> The output is a probability distribution over the next token ID.

That's the entire interface. Everything else — structured prompts, system messages, tool calls — is just formatting conventions that get serialized into this single integer sequence.


---
## 4. Comparing Tokenizers Across Models

Different model families use different tokenizers — trained on different corpora with different vocabulary sizes. This means the same text can produce different token sequences depending on the model.


In [ ]:
PHI4       = "microsoft/Phi-4-mini-instruct"
DEEPSEEK   = "deepseek-ai/DeepSeek-V3.1"
QWEN_CODER = "Qwen/Qwen2.5-Coder-7B-Instruct"


In [ ]:
phi4_tokenizer = AutoTokenizer.from_pretrained(PHI4)

sample_text = "I am curiously excited to show Hugging Face Tokenizers in action to my LLM engineers"

print("=== Llama 3.1 ===")
llama_tokens = tokenizer.encode(sample_text)
print("IDs      :", llama_tokens)
print("Fragments:", tokenizer.batch_decode(llama_tokens))

print("\n=== Phi-4 ===")
phi4_tokens = phi4_tokenizer.encode(sample_text)
print("IDs      :", phi4_tokens)
print("Fragments:", phi4_tokenizer.batch_decode(phi4_tokens))


In [ ]:
# Chat template comparison: Llama vs Phi-4
print("=== Llama 3.1 Chat Template ===")
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

print("\n=== Phi-4 Chat Template ===")
print(phi4_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))


In [ ]:
deepseek_tokenizer = AutoTokenizer.from_pretrained(DEEPSEEK)

print("Token counts for the same sentence:")
print(f"  Llama 3.1 : {len(tokenizer.encode(sample_text))} tokens")
print(f"  Phi-4     : {len(phi4_tokenizer.encode(sample_text))} tokens")
print(f"  DeepSeek  : {len(deepseek_tokenizer.encode(sample_text))} tokens")


In [ ]:
# Full chat template comparison across all three models
print("=== Llama 3.1 ===")
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

print("\n=== Phi-4 ===")
print(phi4_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

print("\n=== DeepSeek ===")
print(deepseek_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))


---
## 5. Tokenizing Code with QwenCoder

Code-focused models like QwenCoder are trained on large programming corpora. Let's see how a code tokenizer segments Python syntax at the token level.


In [ ]:
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_CODER)

code_snippet = '''
def hello_world(person):
    print("Hello", person)
'''

tokens = qwen_tokenizer.encode(code_snippet)

print(f"Code tokenized into {len(tokens)} tokens:\n")
for token_id in tokens:
    decoded = qwen_tokenizer.decode(token_id)
    print(f"  {token_id:>7}  →  {repr(decoded)}")


---
## Summary

| Concept | Description |
|---|---|
| **Token** | A sub-word fragment that is the atomic unit of LLM input |
| **Token ID** | The integer index of a token in the model's vocabulary |
| **Vocabulary** | The complete set of tokens a model knows (~32K–128K tokens) |
| **Chat Template** | A model-specific format that serializes role-based messages into a flat string |
| **`encode()`** | Converts text → list of token IDs |
| **`decode()`** | Converts token IDs → text |
| **`apply_chat_template()`** | Formats messages dict → model-ready prompt string |

Different models use different tokenizers — vocabulary size, special tokens, and chat templates all vary. Always use the tokenizer that matches your model.
